Implementasi dan Optimasi Algoritma Decision Tree, Naïve Bayes, dan K-Nearest Neighbors untuk Deteksi Dini Penyakit Jantung



## Identitas Kelompok

| Nama | NIM |
|------|-----|
| Farel Hagasa Tarigan  | 103032400032 |
| [Nama Anggota 2]      | [NIM 2] |
| [Nama Anggota 3]      | [NIM 3] |

---



## 1. Pendahuluan

### 1.1 Latar Belakang

Penyakit jantung merupakan salah satu penyebab kematian tertinggi di dunia. 
Menurut World Health Organization (WHO), penyakit kardiovaskular menyebabkan 
sekitar 17,9 juta kematian setiap tahunnya, mewakili 32% dari seluruh kematian 
global. Di Indonesia sendiri, penyakit jantung koroner menempati posisi kedua 
penyebab kematian tertinggi berdasarkan data Kementerian Kesehatan Republik 
Indonesia.

Deteksi dini penyakit jantung menjadi sangat krusial mengingat tingginya angka 
kematian yang dapat dicegah apabila penyakit teridentifikasi sejak awal. 
Pendekatan tradisional dalam diagnosis penyakit jantung seringkali membutuhkan 
waktu yang lama, biaya yang tinggi, serta bergantung penuh pada keahlian tenaga 
medis. Oleh karena itu, pemanfaatan teknologi machine learning dalam membantu 
proses diagnosis menjadi solusi yang menjanjikan.

Penelitian ini mengimplementasikan tiga algoritma machine learning yaitu 
**Decision Tree**, **Naïve Bayes**, dan **K-Nearest Neighbors (KNN)** untuk 
mengklasifikasikan kondisi pasien berdasarkan data klinis. Ketiga algoritma 
tersebut dioptimasi menggunakan **GridSearchCV** dan dievaluasi menggunakan 
**Stratified K-Fold Cross Validation** untuk memastikan hasil yang robust dan 
dapat diandalkan.

---

### 1.2 Rumusan Masalah

Berdasarkan latar belakang di atas, rumusan masalah dalam penelitian ini adalah:

1. Bagaimana mengimplementasikan algoritma Decision Tree, Naïve Bayes, dan KNN 
   untuk klasifikasi penyakit jantung?
2. Bagaimana pengaruh hyperparameter tuning terhadap performa masing-masing model?
3. Algoritma manakah yang memberikan performa terbaik dalam mendeteksi penyakit 
   jantung berdasarkan dataset yang digunakan?

---

### 1.3 Tujuan Penelitian

1. Mengimplementasikan dan membandingkan tiga algoritma machine learning dalam 
   konteks klasifikasi penyakit jantung.
2. Mengoptimasi hyperparameter setiap model menggunakan GridSearchCV.
3. Mengevaluasi dan menganalisis performa model secara komprehensif menggunakan 
   metrik yang relevan.

---

### 1.4 Deskripsi Dataset

Dataset yang digunakan dalam penelitian ini adalah **Heart Disease Dataset** 
yang bersumber dari **Kaggle** (Johns Hopkins University / UCI Machine Learning 
Repository). Dataset ini berisi rekam medis pasien yang dikumpulkan untuk 
keperluan penelitian diagnosis penyakit jantung.

**Informasi Dataset:**

| Keterangan | Detail |
|------------|--------|
| Sumber | Kaggle — Heart Disease Dataset |
| Jumlah Data | 1.025 baris |
| Jumlah Fitur | 13 fitur input + 1 target |
| Jenis Masalah | Klasifikasi Biner |
| Target | 0 = Tidak Sakit, 1 = Sakit Jantung |

**Deskripsi Fitur:**

| No | Fitur | Deskripsi |
|----|-------|-----------|
| 1 | age | Usia pasien (tahun) |
| 2 | sex | Jenis kelamin (1=Pria, 0=Wanita) |
| 3 | cp | Tipe nyeri dada (0-3) |
| 4 | trestbps | Tekanan darah saat istirahat (mm Hg) |
| 5 | chol | Kolesterol serum (mg/dl) |
| 6 | fbs | Gula darah puasa > 120 mg/dl (1=Ya, 0=Tidak) |
| 7 | restecg | Hasil elektrokardiografi saat istirahat (0-2) |
| 8 | thalach | Detak jantung maksimum yang dicapai |
| 9 | exang | Angina akibat olahraga (1=Ya, 0=Tidak) |
| 10 | oldpeak | Depresi ST akibat olahraga |
| 11 | slope | Kemiringan segmen ST puncak olahraga |
| 12 | ca | Jumlah pembuluh darah utama (0-3) |
| 13 | thal | Thalassemia (0-3) |
| 14 | target | Diagnosis penyakit jantung (0=Sehat, 1=Sakit) |

---

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
import warnings
warnings.filterwarnings('ignore')

print("✅ Library berhasil diimport")

✅ Library berhasil diimport


In [2]:
# Load Dataset
df = pd.read_csv('heart.csv')

print("=" * 50)
print("INFORMASI DATASET")
print("=" * 50)
print(f"Jumlah baris    : {df.shape[0]}")
print(f"Jumlah kolom    : {df.shape[1]}")

print("\nPreview 5 data pertama:")
display(df.head())

print("\nStatistik Deskriptif:")
display(df.describe().round(2))

print("\nTipe Data:")
display(df.dtypes.to_frame('Tipe Data'))

print("\nMissing Values:")
missing = df.isnull().sum().to_frame('Jumlah Missing')
missing['Persentase'] = (df.isnull().sum() / len(df) * 100).round(2)
display(missing)

print("\nDistribusi Target:")
target_dist = df['target'].value_counts().to_frame('Jumlah')
target_dist['Persentase (%)'] = (df['target'].value_counts(normalize=True) * 100).round(2)
target_dist.index = ['Sakit (1)', 'Sehat (0)']
display(target_dist)

INFORMASI DATASET
Jumlah baris    : 1025
Jumlah kolom    : 14

Preview 5 data pertama:


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0



Statistik Deskriptif:


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
count,1025.00,1025.00,1025.00,1025.00,1025.00,1025.00,1025.00,1025.00,1025.00,1025.00,1025.00,1025.00,1025.00,1025.00
mean,54.43,0.70,0.94,131.61,246.00,0.15,0.53,149.11,0.34,1.07,1.39,0.75,2.32,0.51
std,9.07,0.46,1.03,17.52,51.59,0.36,0.53,23.01,0.47,1.18,0.62,1.03,0.62,0.50
min,29.00,0.00,0.00,94.00,126.00,0.00,0.00,71.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,48.00,0.00,0.00,120.00,211.00,0.00,0.00,132.00,0.00,0.00,1.00,0.00,2.00,0.00
50%,56.00,1.00,1.00,130.00,240.00,0.00,1.00,152.00,0.00,0.80,1.00,0.00,2.00,1.00
75%,61.00,1.00,2.00,140.00,275.00,0.00,1.00,166.00,1.00,1.80,2.00,1.00,3.00,1.00
max,77.00,1.00,3.00,200.00,564.00,1.00,2.00,202.00,1.00,6.20,2.00,4.00,3.00,1.00



Tipe Data:


,Tipe Data
age,int64
sex,int64
cp,int64
trestbps,int64
chol,int64
fbs,int64
restecg,int64
thalach,int64
exang,int64
oldpeak,float64



Missing Values:


,Jumlah Missing,Persentase
age,0,0.0
sex,0,0.0
cp,0,0.0
trestbps,0,0.0
chol,0,0.0
fbs,0,0.0
restecg,0,0.0
thalach,0,0.0
exang,0,0.0
oldpeak,0,0.0



Distribusi Target:


,Jumlah,Persentase (%)
Sakit (1),526,51.32
Sehat (0),499,48.68
